In [1]:
import bpy

print(type(bpy.data))
print(bpy.data.rna_type)

<class 'bpy.types.BlendData'>
<bpy_struct, Struct("BlendData") at 0x70adc00>


In [2]:
print(type(bpy.data.libraries))
print(bpy.data.libraries.rna_type)

<class 'bpy_prop_collection'>
<bpy_struct, Struct("BlendDataLibraries") at 0x7041d00>


In [9]:
data_path = "/home/ik/Dev/blib5@github/assets/lettering/dagger-2026-05-07.blend"
with bpy.data.libraries.load(data_path) as (data_src, data_dst):
    print(dir(data_src))
    print(data_src.objects)

['actions', 'annotations', 'armatures', 'brushes', 'cache_files', 'cameras', 'collections', 'curves', 'fonts', 'grease_pencils', 'hair_curves', 'images', 'lattices', 'lightprobes', 'lights', 'linestyles', 'masks', 'materials', 'meshes', 'metaballs', 'movieclips', 'node_groups', 'objects', 'paint_curves', 'palettes', 'particles', 'pointclouds', 'scenes', 'screens', 'sounds', 'speakers', 'texts', 'textures', 'version', 'volumes', 'workspaces', 'worlds']
['Light', 'dagger', 'Camera']


In [ ]:
import importlib
import sys
n = importlib.import_module('blib5.network')
print(n.__spec__.parent)
globals()['a']=1
print(sys.modules)
m = importlib.import_module('blib5.geo.bezier')
print(m.__spec__.parent)
sys.modules['blib5']


In [10]:
import importlib
import sys
m = importlib.import_module('blib5.geo.bezier')
#print(sys.modules.keys())
sys.modules['blib5'].geo.bezier


<module 'blib5.geo.bezier' from '/home/ik/Dev/blib5@github/blib5/package/src/blib5/geo/bezier.py'>

In [39]:
from blib5.remote import connection
remote = connection()




KeyboardInterrupt: 

In [37]:
remote.import_lib('blib5.mesh')
del remote

NameError: name 'remote' is not defined

In [59]:
import blib5.scene
remote.import_lib('blib5.scene')
collections = [
    'a/e',
    'a/e/x',
    'x/z',
    'y/x'
]
remote.command('blib5.scene.ensure_collections', collections)
blib5.scene.ensure_collections(collections)


In [55]:
s = bpy.data.scenes[0]
print(s.collection.name)
s.collection.children[0].name in s.collection.children
dir(s.collection.children)

Scene Collection


['__bool__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__doc__',
 '__doc__',
 '__getattribute__',
 '__getitem__',
 '__iter__',
 '__len__',
 '__module__',
 '__setattr__',
 '__setitem__',
 '__slots__',
 'bl_rna',
 'find',
 'foreach_get',
 'foreach_set',
 'get',
 'items',
 'keys',
 'link',
 'rna_type',
 'unlink',
 'values']

In [12]:
import sys
import importlib
m = importlib.import_module('blib5.module')
base_module = m.__name__.split('.')[0]
base_module

'blib5'

In [51]:
sys = importlib.import_module('sys')
sys.argv
blib5 = importlib.import_module('blib5')
blib5.curve = importlib.import_module('blib5.curve')
blib5.curve.__dict__
type(blib5.curve)
import types
isinstance(blib5.curve, types.ModuleType)

True

In [ ]:
import numpy as np
import scipy as sp

'''

'''
def binomial_coefficients(M, CACHE={}):
    Ci = CACHE.get(M)
    if Ci is None:
        # comb(N, k, *, exact=False, repetition=False)
        # number of combinations of k items taken out of N items
        Ci = sp.special.comb(M, np.arange(M+1))
        CACHE[M] = Ci
    return Ci

binomial_coefficients(0)

array([1.])

In [ ]:

_missing = object()
def cache_unary_function_result(f):
    def f_wrapper(x, CACHE={}):
        r = CACHE.get(x, _missing)
        if r is _missing:
            r = f(x)
            CACHE[x] = r
        return r
    return f_wrapper

@cache_unary_function_result
def f(x):
    print(f"calculate f {x}")
    return x*x 

@cache_unary_function_result
def g(x):
    print(f"calculate g {x}")
    return x-1

[f(1), f(2), f(1), g(1), g(1)]

calculate f 1
calculate f 2
calculate g 1


[1, 4, 1, 0, 0]

In [1]:
import blib5.geo.spline as spline

spline.bezier_polynomial_coefficients(3)

AttributeError: module 'blib5.geo.spline' has no attribute 'bezier_polynomial_coefficients'

In [12]:
import numpy as np 
from blib5 import decorate

def calculator(R):
    MP1 = R.shape[-2]
    if R.ndim == 2:
        def calculator(U):
            npoints = U.shape[0]
            '''
            construct PU with every row i = [1 u[i] u[i]² u[i]³ ... ]
            U = np.linspace(0,1,5)
            PU = 
                [1.       0.       0.       0.      ]
                [1.       0.25     0.0625   0.015625]
                [1.       0.5      0.25     0.125   ]
                [1.       0.75     0.5625   0.421875]
                [1.       1.       1.       1.      ]]
            '''        
            PU = np.empty((npoints, MP1))
            PU[:] = U[:, np.newaxis]
            PU[:,0] = 1
            PU = np.cumprod(PU, axis=1)
            return PU @ R
    elif R.ndim == 3:
        def calculator(IDX, U):
            npoints = U.shape[0]      
            PU = np.empty((npoints, MP1))
            PU[:] = U[:, np.newaxis]
            PU[:,0] = 1
            PU = np.cumprod(PU, axis=1)
            return (PU[:,:,np.newaxis]*R[IDX]).sum(axis=1) 
        calculator.nsegments = R.shape[0]  
    calculator.R = R
    return calculator


#############################################################################################
'''
Calculate the polynomial coefficients for every control point.

polynomial_coefficients_of_1d_bezier(3)
=>
array([[ 1, -3,  3, -1],
       [ 0,  3, -6,  3],
       [ 0,  0,  3, -3],
       [ 0,  0,  0,  1]], dtype=int32)

'''
@decorate.cache_unary_function_result
def bezier_polynomial_coefficients(M):
    B = np.zeros( (M+1,M+1), dtype=np.int32)
    B[:,-1] = 1
    for i in range(M,0,-1):
        B[i-1] = -B[i]
        B[i-1,:-1] += B[i,1:]
    B = B * np.abs(B[0]).reshape((M+1,1))
    return B


'''
Construct a point calculator for a fixed set of M+1 control points and variable U.
The shape of control points is (N, M+1, dimension of points) for N segments.
'''
def calculator_1d_bezier(CONTROL_POINTS):
    ''' shape of control points is (M+1, dimension of points)'''
    M = CONTROL_POINTS.shape[-2] - 1  # degree of the polynomial
    B = bezier_polynomial_coefficients(M).transpose()
    # we squash all terms of the same degree together to make an efficient calculator that takes a variable array of t values.
    R = B @ CONTROL_POINTS
    return calculator(R)


In [13]:
CP = np.array( [
    [0.0,0],
    [1,1],
    [2,1],
    [3,0],
])
c = calculator_1d_bezier(CP)
c.R.shape

(4, 2)

In [14]:
U = np.linspace(0,1,5)
c(U)

array([[0.    , 0.    ],
       [0.75  , 0.5625],
       [1.5   , 0.75  ],
       [2.25  , 0.5625],
       [3.    , 0.    ]])

In [ ]:
def make_functions():
    sock = None 
    client_count = 0
    def f():
        client_count += 1
        print(f'{client_count=}') 
        if client_count == 1:
            sock = 1 
    def g():
        client_count -= 1 
        print(f'{client_count=}') 
        if client_count == 0:
            sock -= 1 

f, g = make_functions()
